# Integrate orbits and extract frequencies in parallel

This tutorial is useful if you need the frequencies of many orbits. The analysis of one orbit is completely independent from the others, so it can be distributed over the CPU cores. We integrate orbits in the isochrone potential using the package [agama](https://github.com/GalacticDynamics-Oxford/Agama), extract the frequencies in parallel, and compare the fundamental frequencies with the analytical ones.

Two remarks before we start:

* `agama.orbit` already integrates the orbits in parallel (over all cores), so only the frequency analysis needs to be parallelized here.
* With `naif.hunt_peak_freqs` (see the Introduction), the frequency analysis is several times faster than with `naif.find_peak_freqs`. For a few thousand orbits, a simple serial loop may then be fast enough; we compare both options at the end.

As usual, we start importing the relevant modules:

In [1]:
import os
import time
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import agama
import naif

Then we define some relevant numbers, such as number of orbits, points per orbit and so on:

In [2]:
# n. of orbits:
n_orbs = 1_000
# n. of circular periods to integrate for:
n_Tcirc = 100
# n. of points per orbit:
n_steps = 10_000
# n. of orbits integrated and analysed at a time (to limit the memory used):
n_orbs_loop = 112
# n. of frequencies to extract per time-series:
n_freqs = 5
# n. of processes for the frequency analysis:
n_workers = os.cpu_count()
print('workers:', n_workers)

workers: 16


## The parallel strategy

We process the orbits in blocks of `n_orbs_loop`: each block is integrated, its time-series are stored in **global** arrays (`f_time`, `f_r`, `f_phi`), and a pool of `n_workers` processes analyses the orbits of the block, one task per orbit. The pool is created with the `'fork'` start method, so each process starts as a copy of the main one and reads the global arrays directly, without copying them. Since the pool is created after the arrays of the block are filled, the processes always see the current block.

Each task analyses both time-series of one orbit, $f_r = r$ and $f_\varphi = \sqrt{2|L_z|}\,e^{i\varphi}$, with the function `freq_func`, which can be `naif.hunt_peak_freqs` or `naif.find_peak_freqs` (both have the same inputs and outputs):

In [3]:
def analyse_orbit(j):
    om_r, a_r = freq_func(f_r[j], f_time[j], n_freqs=n_freqs, verbose=False)
    om_phi, a_phi = freq_func(f_phi[j], f_time[j], n_freqs=n_freqs, verbose=False)
    return j, om_r, a_r, om_phi, a_phi

The function below loops over the blocks, integrates the orbits, extracts the frequencies (in parallel or not) and stores them for all orbits. It also returns the time spent in the integration and in the frequency analysis:

In [4]:
def run(func, parallel=True):
    global freq_func, f_time, f_r, f_phi
    freq_func = func

    om_r = np.zeros((n_orbs, n_freqs))
    om_phi = np.zeros((n_orbs, n_freqs))
    a_r = np.zeros((n_orbs, n_freqs), dtype=np.complex128)
    a_phi = np.zeros((n_orbs, n_freqs), dtype=np.complex128)
    t_int = t_freq = 0.

    for i in range(0, n_orbs, n_orbs_loop):
        ic = isoc_data[i:i+n_orbs_loop]
        k = len(ic)

        # integrate (agama runs in parallel over the orbits):
        t0 = time.perf_counter()
        orbs = agama.orbit(potential=isoc_pot, ic=ic, time=n_Tcirc*isoc_pot.Tcirc(ic),
                           trajsize=n_steps+1, accuracy=1e-15, dtype=float,
                           separateTime=False, verbose=False)
        t_int += time.perf_counter() - t0

        # time-series of this block (global arrays read by analyse_orbit):
        f_time = np.vstack(orbs[:,0])[:,:-1]
        coords = np.vstack(orbs[:,1]).reshape(k, n_steps+1, 6)[:,:-1]
        x, y, z = coords[...,0], coords[...,1], coords[...,2]
        vx, vy = coords[...,3], coords[...,4]
        f_r = np.sqrt(x**2 + y**2 + z**2)
        f_phi = np.sqrt(2*np.abs(x*vy - y*vx))*np.exp(1j*np.arctan2(y, x))

        # frequency analysis:
        t0 = time.perf_counter()
        if parallel:
            with ProcessPoolExecutor(max_workers=n_workers, mp_context=mp.get_context('fork')) as pool:
                results = list(pool.map(analyse_orbit, range(k), chunksize=2))
        else:
            results = [analyse_orbit(j) for j in range(k)]
        t_freq += time.perf_counter() - t0

        for j, o_r, a_r_j, o_phi, a_phi_j in results:
            om_r[i+j], a_r[i+j], om_phi[i+j], a_phi[i+j] = o_r, a_r_j, o_phi, a_phi_j

    return om_r, a_r, om_phi, a_phi, t_int, t_freq

## Run

We generate the initial conditions (a self-consistent sample of the isochrone model, in units with $G=1$) and run the analysis in parallel with `naif.hunt_peak_freqs`:

In [5]:
M = 1.
Rc = 1.
isoc_pot = agama.Potential(type='Isochrone', mass=M, scaleRadius=Rc)
isoc_df = agama.DistributionFunction(type='QuasiSpherical', potential=isoc_pot, density=isoc_pot)
isoc_data, _ = agama.GalaxyModel(isoc_pot, isoc_df).sample(n_orbs)

om_r, a_r, om_phi, a_phi, t_int, t_freq = run(naif.hunt_peak_freqs, parallel=True)
print(f'integration: {t_int:.2f} s, frequency analysis: {t_freq:.2f} s')

integration: 0.64 s, frequency analysis: 0.64 s


As in the other tutorials, we take the leading frequency of $f_r$ as $\Omega_r$, and the leading frequency of $f_\varphi$ with $|\Omega_\varphi| > |\Omega_r|/2$ as the fundamental $\Omega_\varphi$:

In [6]:
def fundamental(om_r, om_phi):
    Om_r = np.abs(om_r[:,0])
    Om_phi = np.array([o[np.abs(o) > Om_r[i]/2.][0] for i, o in enumerate(om_phi)])
    return Om_r, Om_phi

Om_r, Om_phi = fundamental(om_r, om_phi)

and compare them with the analytical frequencies of the isochrone model (see Binney $\&$ Tremaine 2008):

In [7]:
pos, vel = isoc_data[:,:3], isoc_data[:,3:]
Lvec = np.cross(pos, vel)
L = np.linalg.norm(Lvec, axis=1)
E = isoc_pot.potential(pos) + 0.5*np.sum(vel**2, axis=1)

an_Jr = M/np.sqrt(-2*E) - 0.5*(L + np.sqrt(L**2 + 4*M*Rc))
an_Om_r = M**2/(an_Jr + 0.5*(L + np.sqrt(L**2 + 4*M*Rc)))**3
an_Om_phi = np.sign(Lvec[:,2])*0.5*(1 + L/np.sqrt(L**2 + 4*M*Rc))*an_Om_r

print(f'median relative error, Omega_r:   {np.median(np.abs(Om_r/an_Om_r - 1)):.1e}')
print(f'median relative error, Omega_phi: {np.median(np.abs(Om_phi/an_Om_phi - 1)):.1e}')

median relative error, Omega_r:   1.2e-10
median relative error, Omega_phi: 2.1e-08


The orbits are integrated with high accuracy and stored in double precision (`accuracy=1e-15, dtype=float`), so the errors are small (compare with the isochrone tutorial, where single precision limits them to $\sim 10^{-7}$).

## Comparing options

For comparison, we run the same analysis with `naif.find_peak_freqs` in parallel, and with `naif.hunt_peak_freqs` in a serial loop:

In [8]:
timings = {'hunt_peak_freqs, parallel': t_freq}
_, _, _, _, _, timings['find_peak_freqs, parallel'] = run(naif.find_peak_freqs, parallel=True)
_, _, _, _, _, timings['hunt_peak_freqs, serial'] = run(naif.hunt_peak_freqs, parallel=False)

print(f'integration (agama, parallel): {t_int:.2f} s')
for name, el in timings.items():
    print(f'frequency analysis, {name:26s} {el:6.2f} s')

integration (agama, parallel): 0.64 s
frequency analysis, hunt_peak_freqs, parallel    0.64 s
frequency analysis, find_peak_freqs, parallel    2.52 s
frequency analysis, hunt_peak_freqs, serial      2.89 s


`naif.hunt_peak_freqs` in parallel is the fastest option: on a machine with many cores, the frequency analysis then takes about as long as the orbit integration itself. With `naif.find_peak_freqs`, parallelization is much more important. For moderate numbers of orbits, `naif.hunt_peak_freqs` in a serial loop is comparable to `naif.find_peak_freqs` in parallel, without any of the multiprocessing machinery.

Some comments on the code above:

- **Start method.** The `'fork'` start method lets the processes read the global arrays without copying them. With `'spawn'` (the default on macOS and Windows), the processes cannot see these arrays, and the time-series would have to be sent to each process; in our tests, this made the parallel analysis about 4 times slower with `naif.find_peak_freqs` and more than 10 times slower with `naif.hunt_peak_freqs`. `'fork'` is available on Linux and macOS (not on Windows).
- **A caveat on `'fork'`.** Forking a process that runs several threads is not guaranteed to be safe (`agama` uses OpenMP threads for the orbit integration), and recent Python versions warn about it. It works in our tests, but if a run hangs, this is a possible cause.
- **Scripts.** In a python script, put the code that runs the analysis (here, the calls to `run`) under `if __name__ == "__main__":`. This is required with `'spawn'` and good practice in general.
- **Memory.** The blocks of `n_orbs_loop` orbits limit the memory used by the time-series: a block of 112 orbits with $10^4$ points uses ~150 MB (trajectories and time-series), while all 1000 orbits at once would use more than 1 GB (and more for longer integrations).
- **Pool size.** A new pool is created for each block, since the processes must be forked after the arrays of the block are filled. With `'fork'`, this costs a fraction of a second in total (about 0.25 s for the 9 blocks here); for many more blocks, a larger `n_orbs_loop` reduces it.